# CML Rain Field Reconstruction Pipeline

This is the master project notebook. It starts with the rain-field simulator and will grow as we add sensor maps, measurement simulation, reconstruction, and evaluation.

## Module Flow

The project is organized around stable data contracts:

```text
RainFieldSimulator  ->  RainField
SensorMapGenerator  ->  SensorMap
MeasurementSimulator(RainField, SensorMap)  ->  Measurements + ForwardOperator
ReconstructionModel(Measurements, SensorMap, GridSpec)  ->  ReconstructionResult
Evaluator(ReconstructionResult, RainField)  ->  Metrics
```

At module boundaries we pass physical objects. Inside algorithms we can convert them into control/inverse-problem objects such as `x`, `F`, `H`, and `y`.

## RainField Contract

The rain simulator produces:

```text
RainField:
    values_mm_h: [T, Ny, Nx]
    grid: GridSpec
    time: TimeSpec
    units: mm/hour
    model_name
    model_params
    metadata
```

The grid is cell-centered. A `30 km x 30 km` domain with `dx = dy = 0.5 km` gives `Ny = Nx = 60`.

## Rain Simulator: Advection-Diffusion

The simulator uses the PDE:

```text
dR/dt = -u dR/dx - v dR/dy + D laplacian(R) - lambda R
```

Control/state-space interpretation:

```text
x[k+1] = F x[k]
```

but the implementation is matrix-free: it applies the finite-difference update directly to `R[y, x]` instead of explicitly storing the sparse matrix `F`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt

from rain import (
    AdvectionDiffusionConfig,
    GaussianInitialCondition,
    GridSpec,
    TimeSpec,
    simulate_advection_diffusion_rain,
)
from visualization import animate_rain_field, display_rain_animation

## 1. Configure The Physical Domain And PDE

In [ ]:
grid = GridSpec(
    domain_width_km=30.0,
    domain_height_km=30.0,
    dx_km=0.5,
    dy_km=0.5,
)

time = TimeSpec(
    duration_h=1.0,
    dt_h=1.0 / 60.0,
)

config = AdvectionDiffusionConfig(
    grid=grid,
    time=time,
    wind_velocity_km_h=(8.0, 3.0),
    diffusion_km2_h=0.04,
    decay_h_inv=0.02,
    initial_conditions=(
        GaussianInitialCondition(
            center_x_km=8.0,
            center_y_km=15.0,
            sigma_x_km=1.4,
            sigma_y_km=2.0,
            intensity_mm_h=35.0,
        ),
        GaussianInitialCondition(
            center_x_km=18.0,
            center_y_km=9.0,
            sigma_x_km=2.0,
            sigma_y_km=1.2,
            intensity_mm_h=22.0,
        ),
    ),
)

## 2. Run The Simulator

In [ ]:
rain = simulate_advection_diffusion_rain(config)
rain.summary()

In [ ]:
print("R array shape [T, Y, X]:", rain.values_mm_h.shape)
print("x centers [km]:", rain.x_km[:5], "...")
print("y centers [km]:", rain.y_km[:5], "...")
print("time [h]:", rain.t_h[:5], "...")
print("model:", rain.model_name)
print("PDE:", rain.metadata["pde"])
print("state-space view:", rain.metadata["state_space_view"])

## 3. Plot One Frame

In [ ]:
frame_index = 20

plt.figure(figsize=(6, 5))
plt.imshow(
    rain.at_grid(frame_index),
    origin="lower",
    extent=[
        rain.grid.x_edges_km.min(),
        rain.grid.x_edges_km.max(),
        rain.grid.y_edges_km.min(),
        rain.grid.y_edges_km.max(),
    ],
    cmap="Blues",
    vmin=0.0,
    vmax=rain.values_mm_h.max(),
)
plt.colorbar(label="Rain rate [mm/hour]")
plt.xlabel("x [km]")
plt.ylabel("y [km]")
plt.title(f"Advection-diffusion rain at t = {rain.t_h[frame_index]:.2f} h")
plt.show()

## 4. Play The Rain Field Video

In [ ]:
animation = animate_rain_field(rain, interval_ms=120, title="Advection-diffusion rain")
display_rain_animation(animation)

## Next Interfaces To Implement

### SensorMap

```text
cml_links:
    id, start/end coordinates, length, frequency, polarization, a, b, noise model

weather_stations:
    id, point coordinate, sensor type, noise/bias/missingness model

personal_weather_stations:
    same point interface, usually noisier and denser
```

### Measurements

```text
timestamp, sensor_id, sensor_type, measurement_type, value, unit,
clean_value, noise_value, quality_flag
```

### ForwardOperator

```text
H_cml, H_ws, H_pws, sensor_order, grid_shape
```

The CML measurement model can start linear with path-averaged rain and later become nonlinear attenuation:

```text
A_i(t) = integral_link a_i R(x,y,t)^b_i dl + noise
```